# Golf Score Predictor — Model Training
Synthetic data for demonstration only. The test error measures fit to simulated scores, not real-world accuracy.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

In [2]:
# Generate synthetic golf statistics for 2,000 example rounds.
rng = np.random.default_rng(42)
size = 2000
handicap = rng.uniform(-5, 54, size).round(1)
distance = np.clip(280 - 2 * handicap + rng.normal(0, 20, size), 100, 350).round().astype(int)
fairways = np.clip(np.rint(9 - 0.12 * handicap + rng.normal(0, 2, size)), 0, 14).astype(int)
greens = np.clip(np.rint(12 - 0.25 * handicap + rng.normal(0, 2, size)), 0, 18).astype(int)
putts = np.clip(np.rint(30 + 0.22 * handicap + rng.normal(0, 3, size)), 18, 60).astype(int)
score = np.clip(np.rint(72 + 0.65 * handicap + 0.8 * (10 - greens) + 0.7 * (putts - 32) + 0.3 * (7 - fairways) + 0.025 * (250 - distance) + rng.normal(0, 3, size)), 55, 160).astype(int)
data = pd.DataFrame({'driving_distance': distance, 'fairways_hit': fairways, 'greens_in_regulation': greens, 'total_putts': putts, 'handicap': handicap, 'score': score})
data.to_csv('golf_data.csv', index=False)
print(data.head())

   driving_distance  fairways_hit  ...  handicap  score
0               178             0  ...      40.7    115
1               256             5  ...      20.9     93
2               165             5  ...      45.7    125
3               202             3  ...      36.1    109
4               267             7  ...       0.6     62

[5 rows x 6 columns]


In [3]:
# Train and evaluate the Random Forest model.
X = data.drop(columns=['score'])
y = data['score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print('Mean absolute error:', round(mean_absolute_error(y_test, model.predict(X_test)), 2))
joblib.dump(model, 'golf_model.joblib')

Mean absolute error: 2.59


['golf_model.joblib']

In [4]:
# Test the same input used in the prototype sketch.
example = pd.DataFrame([[230, 6, 4, 36, 25]], columns=X.columns)
print('Predicted score:', round(model.predict(example)[0]))

Predicted score: 97
